# Activité Pratique N°5 — Partie B1 : Bases de LangGraph


## Configuration


In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
MODEL_NAME = "openai/gpt-oss-20b"

assert GROQ_API_KEY, "Ajoutez GROQ_API_KEY dans partie5/agentic-mcp/.env"
llm = ChatOpenAI(model=MODEL_NAME, api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL, temperature=0)
print(f"Configuration OK | Modèle Groq : {MODEL_NAME}")

Configuration OK | Modèle Groq : openai/gpt-oss-20b


## 1. État et Graphe simple


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class SimpleState(TypedDict):
    nom: str
    message: str

def saluer(state: SimpleState) -> dict:
    return {"message": f"Bonjour {state['nom']}, bienvenue dans LangGraph !"}

builder = StateGraph(SimpleState)
builder.add_node("saluer", saluer)
builder.add_edge(START, "saluer")
builder.add_edge("saluer", END)

graphe_simple = builder.compile()
res = graphe_simple.invoke({"nom": "Mohamed", "message": ""})
print(res)

{'nom': 'Mohamed', 'message': 'Bonjour Mohamed, bienvenue dans LangGraph !'}


## 2. Routage conditionnel


In [3]:
from typing import Literal

class NoteState(TypedDict):
    note: float
    decision: str

def evaluer(state: NoteState) -> dict:
    return {}

def valider(state: NoteState) -> dict:
    return {"decision": f"Admis avec {state['note']}/20"}

def rattrapage(state: NoteState) -> dict:
    return {"decision": f"Rattrapage requis ({state['note']}/20)"}

def router_note(state: NoteState) -> Literal["valider", "rattrapage"]:
    return "valider" if state["note"] >= 10 else "rattrapage"

g_cond = StateGraph(NoteState)
g_cond.add_node("evaluer", evaluer)
g_cond.add_node("valider", valider)
g_cond.add_node("rattrapage", rattrapage)
g_cond.add_edge(START, "evaluer")
g_cond.add_conditional_edges("evaluer", router_note)
g_cond.add_edge("valider", END)
g_cond.add_edge("rattrapage", END)

app_cond = g_cond.compile()
print("Note 15 :", app_cond.invoke({"note": 15, "decision": ""})["decision"])
print("Note 8  :", app_cond.invoke({"note": 8, "decision": ""})["decision"])

Note 15 : Admis avec 15/20
Note 8  : Rattrapage requis (8/20)


## 3. Boucles


In [4]:
class LoopState(TypedDict):
    compteur: int
    historique: list[int]

def incrementer(state: LoopState) -> dict:
    nouveau = state["compteur"] + 1
    return {"compteur": nouveau, "historique": state["historique"] + [nouveau]}

def continuer_boucle(state: LoopState) -> Literal["incrementer", "__end__"]:
    return "incrementer" if state["compteur"] < 4 else "__end__"

g_loop = StateGraph(LoopState)
g_loop.add_node("incrementer", incrementer)
g_loop.add_edge(START, "incrementer")
g_loop.add_conditional_edges("incrementer", continuer_boucle)

app_loop = g_loop.compile()
print(app_loop.invoke({"compteur": 0, "historique": []}))

{'compteur': 4, 'historique': [1, 2, 3, 4]}


## 4. Agent avec Tools et Mémoire


In [5]:
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

@tool
def multiplier(a: int, b: int) -> int:
    """Multiplie deux entiers a et b."""
    return a * b

outils_lg = [multiplier]
llm_avec_outils = llm.bind_tools(outils_lg)

def noeud_agent(state: MessagesState):
    return {"messages": [llm_avec_outils.invoke(state["messages"])]}

g_agent = StateGraph(MessagesState)
g_agent.add_node("agent", noeud_agent)
g_agent.add_node("tools", ToolNode(outils_lg))
g_agent.add_edge(START, "agent")
g_agent.add_conditional_edges("agent", tools_condition)
g_agent.add_edge("tools", "agent")

app_agent = g_agent.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "lg-mem-1"}}

r1 = app_agent.invoke({"messages": [("user", "Combien font 12 * 7 ?")]}, config=cfg)
print("Tour 1 :", r1["messages"][-1].content)

r2 = app_agent.invoke({"messages": [("user", "Multiplie ce résultat par 2.")]}, config=cfg)
print("Tour 2 :", r2["messages"][-1].content)

Tour 1 : 12 × 7 = 84.
Tour 2 : 84 × 2 = 168.


## 5. Human-in-the-Loop


In [6]:
from langgraph.types import interrupt, Command

class HitlState(TypedDict):
    action: str
    statut: str

def preparer_action(state: HitlState) -> dict:
    reponse_humain = interrupt({"question": f"Approuver l'action : {state['action']} ?"})
    if reponse_humain == "oui":
        return {"statut": f"Action exécutée : {state['action']}"}
    return {"statut": "Action annulée par l'utilisateur."}

g_hitl = StateGraph(HitlState)
g_hitl.add_node("preparer", preparer_action)
g_hitl.add_edge(START, "preparer")
g_hitl.add_edge("preparer", END)

app_hitl = g_hitl.compile(checkpointer=InMemorySaver())
cfg_h = {"configurable": {"thread_id": "lg-hitl-1"}}

pause = app_hitl.invoke({"action": "Supprimer les anciens logs", "statut": "en_attente"}, config=cfg_h)
print("Pause HITL :", pause["__interrupt__"][0].value)

reprise = app_hitl.invoke(Command(resume="oui"), config=cfg_h)
print("Après reprise :", reprise["statut"])

Pause HITL : {'question': "Approuver l'action : Supprimer les anciens logs ?"}
Après reprise : Action exécutée : Supprimer les anciens logs


## 6. Multi-Agent System (MAS)


In [7]:
class MasState(TypedDict):
    sujet: str
    notes: str
    resume: str

def agent_chercheur(state: MasState) -> dict:
    rep = llm.invoke(f"Donne 3 points clés très courts sur : {state['sujet']}")
    return {"notes": rep.content}

def agent_redacteur(state: MasState) -> dict:
    rep = llm.invoke(f"Rédige un paragraphe clair (3 phrases max) à partir de ces notes :\n{state['notes']}")
    return {"resume": rep.content}

g_mas = StateGraph(MasState)
g_mas.add_node("chercheur", agent_chercheur)
g_mas.add_node("redacteur", agent_redacteur)
g_mas.add_edge(START, "chercheur")
g_mas.add_edge("chercheur", "redacteur")
g_mas.add_edge("redacteur", END)

app_mas = g_mas.compile()
res_mas = app_mas.invoke({"sujet": "Le protocole MCP en IA agentique", "notes": "", "resume": ""})
print("Notes du chercheur :\n", res_mas["notes"])
print("\nRésumé du rédacteur :\n", res_mas["resume"])

Notes du chercheur :
 - **Structure claire** : MCP définit un schéma de messages, d’états et de rôles pour chaque agent.  
- **Coordination décentralisée** : les agents négocient et résolvent les conflits sans autorité centrale.  
- **Adaptabilité et scalabilité** : le protocole s’ajuste dynamiquement aux changements de l’environnement et supporte un grand nombre d’agents.

Résumé du rédacteur :
 Le protocole MCP impose une structure claire, définissant un schéma précis de messages, d’états et de rôles pour chaque agent. Il favorise une coordination décentralisée, où les agents négocient et résolvent les conflits sans autorité centrale. Grâce à cette architecture, MCP s’adapte dynamiquement aux changements de l’environnement et reste scalable, supportant efficacement un grand nombre d’agents.
